In [1]:
### imports and config

import time
import re
from datetime import datetime

import pandas as pd
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    WebDriverException,
    TimeoutException,
    NoSuchElementException,
    StaleElementReferenceException,
)
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# persistent HTTP session (reuses TCP connections, sends browser headers)
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})

# columns tracked during scraping - box_score_link is internal and gets dropped later
STANDARD_COLUMNS = [
    'season',
    'game_number',
    'date',
    'start_time',
    'opponent',
    'tournament',
    'venue',
    'location_city',
    'promotion',
    'result',
    'unc_score',
    'opponent_score',
    'attendance',
    'box_score_link',
]

REQUEST_DELAY = 0.3




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Adam\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\Adam\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\Adam\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "c:\Users\Adam\anaconda3\Lib\site-packages

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Adam\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\Adam\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\Adam\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "c:\Users\Adam\anaconda3\Lib\site-packages

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



In [2]:
### browser setup (headless chrome via selenium)

def setup_driver():
    opts = webdriver.ChromeOptions()
    for arg in [
        "--headless",
        "--no-sandbox",
        "--disable-dev-shm-usage",
        "--disable-gpu",
        "--window-size=1920,1080",
    ]:
        opts.add_argument(arg)
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts,
    )



In [3]:
### helper functions (validation and cleaning)

### rejects garbage venue values like win-loss records, dates, HTML artifacts
def _valid_venue(v):
    if not v:
        return None
    v = v.strip()
    if not v or len(v) < 3:
        return None
    if re.match(r'^\d+-\d+', v):
        return None
    if re.match(r'^(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d+', v):
        return None
    reject_exact = {
        'noscript', 'ncaa tournament', 'tba', 'tbd', 'neutral',
        'home', 'away', 'final', 'postponed', 'canceled', 'cancelled',
    }
    if v.lower() in reject_exact:
        return None
    if re.match(r'^[A-Z@\s]{2,10}$', v) and '@' in v:
        return None
    if re.match(r'^[WLT]\s+\d+-\d+', v):
        return None
    if any(tok in v for tok in ['data-v-', 's-text-', '<', '>', '=\"', '" data-', 'class=', '{', '}']):
        return None
    if re.match(r'^[\d,\s.]+$', v):
        return None
    if re.match(r'^[A-Z]{1,4}$', v):
        return None
    return v


### strips commas and text from attendance, returns plain integer string or None
def _clean_attendance(v):
    if not v:
        return None
    v = str(v).strip().replace(',', '').replace('.', '')
    m = re.search(r'(\d+)', v)
    if m:
        num = m.group(1)
        if 0 < int(num) <= 100000:
            return num
    return None


### returns True if the result field is missing or a placeholder
def _needs_result_fix(result_text):
    if result_text is None:
        return True
    r = str(result_text).strip()
    return r == '' or r == '-' or r.lower() in {'tbd', 'tba', 'ppd', 'ppd.'}


### strips AP/coaches poll rankings from opponent names
def _clean_opponent(opp):
    if not opp:
        return opp
    opp = str(opp).strip()
    opp = re.sub(r'^#\d+\s+', '', opp)
    opp = re.sub(r'^No\.?\s*\d+(?:/\d+)?\s+', '', opp)
    return opp.strip()


### pulls tournament name from parentheses in the opponent field
def _extract_tournament_from_opponent(opp):
    if not opp:
        return opp, None
    opp = str(opp).strip()
    m = re.match(r'^(.+?)\s*\(([^)]+)\)\s*$', opp)
    if m:
        clean = m.group(1).strip()
        paren = m.group(2).strip()
        skip_patterns = ['continuation', 'ppd', 'makeup', 'suspended', 'resumed']
        if any(sp in paren.lower() for sp in skip_patterns):
            return clean, None
        if re.match(r'^(?:ACC|SEC|Big (?:Ten|12|East)|Pac[- ]?\d+|AAC|Big South|CAA|SoCon|A-?10|Big West|MWC|WAC|Sun Belt)$', paren, re.IGNORECASE):
            return clean, None
        return clean, paren
    return opp, None


### pulls tournament name from parentheses or "at ..." in the location field
def _extract_tournament_from_location(loc):
    if not loc:
        return loc, None
    loc = str(loc).strip()

    m = re.match(r'^(.+?)\s*\(([^)]+)\)\s*$', loc)
    if m:
        clean = m.group(1).strip()
        paren = m.group(2).strip()
        tournament_keywords = ['tournament', 'tourney', 'regional', 'world series',
                               'classic', 'blast', 'invitational', 'challenge']
        if any(kw in paren.lower() for kw in tournament_keywords):
            return clean, paren
        venue_keywords = ['stadium', 'field', 'park', 'ballpark', 'diamond',
                          'dbap', 'bb&t', 'knights', 'us bank']
        if any(kw in paren.lower() for kw in venue_keywords):
            return clean, None
        if re.match(r'^(?:ACC|NCAA|SEC|Big \d+)\b', paren):
            return clean, paren
        return clean, paren

    m2 = re.match(r'^(.+?)\s+at\s+(.+?(?:Tournament|Tourney|Classic|Invitational|Challenge).*)$', loc, re.IGNORECASE)
    if m2:
        return m2.group(1).strip(), m2.group(2).strip()

    return loc, None


### last-resort venue lookup by city when all other sources fail
def _venue_fallback(location_city, season):
    if not location_city:
        return None
    loc = str(location_city).strip().lower()
    loc = re.sub(r'\s*\([^)]*\)', '', loc).strip()
    loc = re.sub(r'\s+at\s+.*(?:tournament|classic|invitational).*$', '', loc, flags=re.IGNORECASE).strip()

    if 'chapel hill' in loc or 'chaple hill' in loc:
        return 'Boshamer Stadium'
    if 'durham' in loc:        return 'Durham Bulls Athletic Park'
    if 'cary' in loc:          return 'USA Baseball NTC'
    if 'charlotte' in loc:     return 'BB&T Ballpark' if season < 2020 else 'Truist Field'
    if 'greensboro' in loc:    return 'NewBridge Bank Park' if season < 2019 else 'First National Bank Field'
    if 'fort mill' in loc:     return 'Knights Stadium' if season < 2014 else 'BB&T Ballpark'
    if 'fayetteville' in loc:  return 'Segra Stadium'
    if 'shelby' in loc:        return 'Veterans Field'
    if 'asheville' in loc:     return 'McCormick Field'
    if 'raleigh' in loc:             return 'Doak Field'
    if 'greenville' in loc:          return 'Clark-LeClair Stadium'
    if 'clemson' in loc:             return 'Doug Kingsmore Stadium'
    if 'tallahassee' in loc:         return 'Dick Howser Stadium'
    if 'blacksburg' in loc:          return 'English Field'
    if 'charlottesville' in loc:     return 'Disharoon Park'
    if 'coral gables' in loc or 'miami' in loc:
        return 'Mark Light Field'
    if 'college park' in loc or loc == 'maryland':
        return 'Shipley Field'
    if 'winston' in loc:
        return 'Gene Hooks Field' if season < 2017 else 'David F. Couch Ballpark'
    if 'pittsburgh' in loc:     return 'Charles L. Cost Field'
    if 'louisville' in loc:     return 'Patterson Stadium'
    if 'atlanta' in loc:        return 'Mac Nease Park'
    if 'chestnut hill' in loc or 'brighton' in loc:
        return 'Harrington Field'
    if 'south bend' in loc or 'notre dame' in loc:
        return 'Frank Eck Stadium'
    if 'columbia' in loc:       return 'Sarge Frye Field'
    if 'wilmington' in loc:   return 'Brooks Field'
    if 'elon' in loc:         return 'Latham Park'
    if 'buies creek' in loc:  return 'Jim Perry Stadium'
    if 'boone' in loc:        return 'Beaver Field'
    if 'rock hill' in loc:    return 'Winthrop Ballpark'
    if 'mount pleasant' in loc: return 'Patriots Point'
    if 'conway' in loc:       return 'Springs Brooks Stadium'
    if 'florence' in loc:     return 'Sparrow Stadium'
    if 'norfolk' in loc:      return 'Bud Metheny Baseball Complex'
    if 'lynchburg' in loc:    return 'Liberty Baseball Stadium'
    if 'gainesville' in loc:  return 'Florida Ballpark' if season >= 2020 else 'McKethan Stadium'
    if 'myrtle beach' in loc: return 'Pelicans Ballpark'
    if 'berkeley' in loc:     return 'Evans Diamond'
    if 'palo alto' in loc:    return 'Sunken Diamond'
    if 'los angeles' in loc:  return 'Jackie Robinson Stadium'
    if 'fresno' in loc:       return 'Pete Bieden Field'
    if 'fullerton' in loc:    return 'Goodwin Field'
    if 'charleston' in loc:   return 'Riley Park'
    if 'richmond' in loc:     return 'The Diamond'
    if 'auburn' in loc:       return 'Plainsman Park'
    if 'tuscaloosa' in loc:   return 'Sewell-Thomas Stadium'
    if 'starkville' in loc:   return 'Dudy Noble Field'
    if 'savannah' in loc:     return 'Historic Grayson'
    if 'norman' in loc:       return 'L Dale Mitchell Park'
    if 'lubbock' in loc:      return 'Rip Griffin Park'
    if 'terre haute' in loc:  return 'Bob Warn Field'
    if 'boca raton' in loc:   return 'FAU Stadium'
    if 'tampa' in loc:        return 'USF Baseball Stadium'
    if 'st. petersburg' in loc: return 'Florida Power Park'
    if 'houston' in loc:      return "Rice's Reckling Park"
    if 'jacksonville' in loc: return 'Baseball Grounds'
    if 'montclair' in loc:    return 'Yogi Berra Stadium'
    if 'salem' in loc:        return 'Salem Memorial Stadium'
    if 'omaha' in loc:
        return 'Rosenblatt Stadium' if season <= 2010 else ('TD Ameritrade Park' if season <= 2021 else 'Charles Schwab Field')
    if 'orlando' in loc:      return 'Disney Wide World of Sports'
    if 'mobile' in loc:       return 'Eddie Stanky Field'
    if 'minneapolis' in loc:  return 'U.S. Bank Stadium'

    return None



In [4]:
### box score page parsing (requests-based, era 1 primary / era 2 fallback)
# era 1 (1999-2016): static HTML stats pages fetched with requests
# era 2 (2017+): used as fallback when selenium dropdown is missing data

### parses final score from box score HTML - tries multiple patterns
def _parse_final_from_stats_html(html):
    out = {}
    try:
        text = BeautifulSoup(html, 'html.parser').get_text('\n')
    except Exception:
        text = re.sub(r'<[^>]+>', ' ', html)

    # pattern 1: "North Carolina 8, Opponent 3"
    m = re.search(
        r'\bNorth Carolina\s+(\d+)\s*,\s*([^\n<]+?)\s+(\d+)\s*(?:\(([^)]+)\))?',
        text, flags=re.IGNORECASE,
    )
    if m:
        unc, opp = m.group(1), m.group(3)
        out.update({'unc_score': unc, 'opponent_score': opp})
        try:
            out['result'] = ('W' if int(unc) > int(opp) else 'L') + f', {unc}-{opp}'
        except ValueError:
            pass
        return out

    # pattern 1b: "Opponent 5, North Carolina 3" (reversed)
    m = re.search(
        r'([^\n<,]+?)\s+(\d+)\s*,\s*North Carolina\s+(\d+)\s*(?:\(([^)]+)\))?',
        text, flags=re.IGNORECASE,
    )
    if m:
        opp_score, unc = m.group(2), m.group(3)
        out.update({'unc_score': unc, 'opponent_score': opp_score})
        try:
            out['result'] = ('W' if int(unc) > int(opp_score) else 'L') + f', {unc}-{opp_score}'
        except ValueError:
            pass
        return out

    # pattern 2: score-by-innings table row
    m_unc = re.search(
        r'^\s*North Carolina\.*.*?-\s+(\d+)\s+\d+\s+\d+\s*$',
        text, flags=re.IGNORECASE | re.MULTILINE,
    )
    if m_unc:
        unc = m_unc.group(1)
        other_lines = re.findall(
            r"^\s*(?!North Carolina)([A-Za-z0-9# .&'\-]+?)\.*.*?-\s+(\d+)\s+\d+\s+\d+\s*$",
            text, flags=re.IGNORECASE | re.MULTILINE,
        )
        if other_lines:
            opp = other_lines[0][1]
            out.update({'unc_score': unc, 'opponent_score': opp})
            try:
                out['result'] = ('W' if int(unc) > int(opp) else 'L') + f', {unc}-{opp}'
            except ValueError:
                pass
            return out

    # pattern 3: HTML table parsing fallback
    try:
        soup = BeautifulSoup(html, 'html.parser')
        lines = [ln.strip() for ln in soup.get_text('\n').splitlines() if ln.strip()]
        unc_total = None
        opp_total = None
        for line in lines:
            if re.search(r'north carolina', line, re.IGNORECASE):
                nums = re.findall(r'\d+', line)
                if len(nums) >= 3:
                    unc_total = nums[-3]
            elif unc_total is not None and opp_total is None:
                nums = re.findall(r'\d+', line)
                if len(nums) >= 3 and not re.search(r'north carolina', line, re.IGNORECASE):
                    opp_total = nums[-3]
                    break
        if unc_total is not None and opp_total is not None:
            out.update({'unc_score': unc_total, 'opponent_score': opp_total})
            try:
                out['result'] = ('W' if int(unc_total) > int(opp_total) else 'L') + f', {unc_total}-{opp_total}'
            except ValueError:
                pass
            return out
    except Exception:
        pass

    return out


### extracts venue from the stats page header (e.g. "March 15, 2008 at Chapel Hill, N.C. (Boshamer Stadium)")
def _parse_venue_from_stats_html(html):
    if not html:
        return None
    soup = BeautifulSoup(html, 'html.parser')
    lines = [ln.strip() for ln in soup.get_text('\n').splitlines() if ln.strip()]
    head = '\n'.join(lines[:250])

    months = r'(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'

    m = re.search(
        rf'\b{months}\.?\s+\d{{1,2}}\s*,\s*\d{{4}}\s+at\s+[^\n(]{{2,220}}(?:\n|\s)*\(\s*([^\)\n]{{2,220}})\s*\)',
        head, flags=re.IGNORECASE,
    )
    if m:
        v = m.group(1).strip()
        if _valid_venue(v):
            return v

    m2 = re.search(
        rf'\b{months}\.?\s+\d{{1,2}}\s*,\s*\d{{4}}\s+at\s+[^\n]{{2,220}}(?:\n|\s)*\(\s*([^\)\n]{{2,220}})\s*\)',
        head, flags=re.IGNORECASE,
    )
    if m2:
        v = m2.group(1).strip()
        if _valid_venue(v):
            return v

    for label in ('Site:', 'Stadium:', 'Venue:', 'Location:'):
        m3 = re.search(rf'{label}\s*([^\n]{{3,120}})', head, re.IGNORECASE)
        if m3:
            v = m3.group(1).strip().rstrip('.')
            if _valid_venue(v):
                return v
    return None


### fallback: looks for explicit venue/site/field labels in the stats page
def _parse_venue_from_stats_explicit_label(html):
    if not html:
        return None
    soup = BeautifulSoup(html, 'html.parser')
    lines = [ln.strip() for ln in soup.get_text('\n').splitlines() if ln.strip()]
    for i, line in enumerate(lines):
        if any(lab in line.lower() for lab in ['stadium:', 'site:', 'venue:', 'field:']):
            if ':' in line:
                val = line.split(':', 1)[-1].strip()
                if _valid_venue(val):
                    return val
            if i + 1 < len(lines):
                val = lines[i + 1].strip()
                if _valid_venue(val):
                    return val
    return None


### extracts "City, State" from stats page header
def _parse_location_from_stats_html(html):
    if not html:
        return None
    soup = BeautifulSoup(html, 'html.parser')
    lines = [ln.strip() for ln in soup.get_text('\n').splitlines() if ln.strip()]
    head = '\n'.join(lines[:200])
    months = r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
    m = re.search(
        rf'\b{months}\.?\s+\d{{1,2}}\s*,\s*\d{{4}}\s+at\s+([A-Z][^\n(]{{2,80}})',
        head, flags=re.IGNORECASE,
    )
    if m:
        loc = m.group(1).strip().rstrip('.')
        if ',' in loc and len(loc) < 60:
            return loc
    return None


### extracts opponent name from stats page (rare fallback for early seasons)
def _parse_opponent_from_stats_html(html):
    if not html:
        return None
    try:
        soup = BeautifulSoup(html, 'html.parser')
        title_tag = soup.find('title')
        if title_tag:
            title = title_tag.get_text().strip()
            m = re.match(r'^(.+?)\s*-\s*Stats\s*-', title, re.IGNORECASE)
            if m:
                opp = re.sub(r'\s*\([^)]*\)\s*$', '', m.group(1).strip()).strip()
                if opp and len(opp) > 1 and 'north carolina' not in opp.lower():
                    return opp
        text = soup.get_text('\n')
        head = '\n'.join([ln.strip() for ln in text.splitlines() if ln.strip()][:100])
        m = re.search(r'North Carolina\s+\d+\s*,\s*([A-Za-z][A-Za-z .&\'\-()]+?)\s+\d+', head, re.IGNORECASE)
        if m:
            opp = m.group(1).strip().rstrip('.')
            if len(opp) > 2 and 'north carolina' not in opp.lower():
                return opp
        m = re.search(r'([A-Za-z][A-Za-z .&\'\-()]+?)\s+\d+\s*,\s*North Carolina\s+\d+', head, re.IGNORECASE)
        if m:
            opp = m.group(1).strip().rstrip('.')
            if len(opp) > 2:
                return opp
    except Exception:
        pass
    return None


### extracts start time from stats/box score page
def _parse_start_time_from_stats_html(html):
    if not html:
        return None

    m = re.search(r'(?:Start Time|Start|First pitch)[:\s]+([\d]{1,2}[:.]\d{2}\s*(?:AM|PM|am|pm|a\.m\.|p\.m\.))', html, re.IGNORECASE)
    if m:
        t = m.group(1).strip().replace('.', '').upper()
        t = re.sub(r'(\d)(AM|PM)', r'\1 \2', t)
        return t

    m = re.search(r'(?:Start Time|Start|First pitch)[:\s]+(\d{1,2}[:.]\d{2})(?:\s|$|\n|<)', html, re.IGNORECASE)
    if m:
        t = m.group(1).strip().replace('.', ':')
        hr = int(t.split(':')[0])
        suffix = 'AM' if 10 <= hr < 12 else 'PM'
        return f'{t} {suffix}'

    m = re.search(r'"start_time"\s*:\s*"([^"]+)"', html, re.IGNORECASE)
    if m:
        return m.group(1).strip()

    m = re.search(r'"startTime"\s*:\s*"([^"]+)"', html, re.IGNORECASE)
    if m:
        return m.group(1).strip()

    m = re.search(r'"gameTime"\s*:\s*"([^"]+)"', html, re.IGNORECASE)
    if m:
        t = m.group(1).strip().replace('.', '').replace('  ', ' ').upper()
        return re.sub(r'(\d)(AM|PM)', r'\1 \2', t)

    try:
        clean = BeautifulSoup(html, 'html.parser').get_text('\n')
    except Exception:
        clean = re.sub(r'<[^>]+>', '\n', html)
    m_date = re.search(r'\d{1,2}/\d{1,2}/\d{4}', clean)
    if m_date:
        after_date = clean[m_date.end():m_date.end() + 500]
        for ln in [ln.strip() for ln in after_date.split('\n') if ln.strip()][:5]:
            mt = re.match(r'^(\d{1,2}[:.]\d{2}\s*(?:AM|PM|am|pm))$', ln, re.IGNORECASE)
            if mt:
                t = mt.group(1).strip().replace('.', ':').upper()
                return re.sub(r'(\d)(AM|PM)', r'\1 \2', t)
            mt = re.match(r'^(\d{1,2}[:.]\d{2})$', ln)
            if mt:
                t = mt.group(1).replace('.', ':')
                hr = int(t.split(':')[0])
                return f'{t} {"AM" if 10 <= hr < 12 else "PM"}'

    return None


### extracts date from stats page header and formats as "Mon D(Day)"
def _parse_date_from_stats_html(html):
    if not html:
        return None
    months_pattern = r'(Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
    m = re.search(rf'{months_pattern}\.?\s+(\d{{1,2}})\s*,\s*(\d{{4}})', html, re.IGNORECASE)
    if m:
        month_str = m.group(1)
        day = int(m.group(2))
        year = int(m.group(3))
        month_map = {
            'jan': 'Jan', 'january': 'Jan', 'feb': 'Feb', 'february': 'Feb',
            'mar': 'Mar', 'march': 'Mar', 'apr': 'Apr', 'april': 'Apr',
            'may': 'May', 'jun': 'Jun', 'june': 'Jun', 'jul': 'Jul', 'july': 'Jul',
            'aug': 'Aug', 'august': 'Aug', 'sep': 'Sep', 'sept': 'Sep', 'september': 'Sep',
            'oct': 'Oct', 'october': 'Oct', 'nov': 'Nov', 'november': 'Nov',
            'dec': 'Dec', 'december': 'Dec',
        }
        month_abbr = month_map.get(month_str.lower(), month_str[:3].title())
        try:
            from datetime import date as dt_date
            month_num = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
                         'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}[month_abbr]
            d = dt_date(year, month_num, day)
            return f'{month_abbr} {day}({d.strftime("%a")})'
        except (ValueError, KeyError):
            return f'{month_abbr} {day}'
    return None


### main box score fetcher - called during scraping to pull all data from a stats page
def get_box_score_details(url):
    info = {}
    if not url or str(url).lower().endswith('.pdf'):
        return info

    try:
        resp = SESSION.get(url, timeout=25)
        if resp.status_code != 200:
            return info
        html = resp.text

        m = re.search(r'Attendance[:\s]+([\d,]+)', html, re.IGNORECASE)
        if m:
            info['attendance'] = m.group(1).replace(',', '')

        st = _parse_start_time_from_stats_html(html)
        if st:
            info['start_time'] = st

        dt = _parse_date_from_stats_html(html)
        if dt:
            info['date'] = dt

        venue = _parse_venue_from_stats_html(html)
        if not venue:
            venue = _parse_venue_from_stats_explicit_label(html)
        if venue:
            info['venue'] = venue

        loc = _parse_location_from_stats_html(html)
        if loc:
            info['location_city'] = loc

        opp = _parse_opponent_from_stats_html(html)
        if opp:
            info['opponent'] = opp

        info.update(_parse_final_from_stats_html(html))

        time.sleep(REQUEST_DELAY)

    except Exception:
        pass

    return info



In [5]:
### game info dropdown (selenium, era 2: 2017+)
# clicks the "Game Info" expander on each game card to get attendance and venue

def get_game_info_details(driver, card):
    attendance = None
    venue = None

    try:
        btn = card.find_element(
            By.CSS_SELECTOR,
            "[data-test-id='s-game-card-standard__header-game-info-expand-icon-button-control']",
        )
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
        time.sleep(0.4)
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(1.0)

        try:
            text = card.find_element(By.CSS_SELECTOR, '.s-game-card__content').text
            lines = text.split('\n')

            for i, line in enumerate(lines):
                low = line.lower().strip()

                if 'attendance' in low:
                    if ':' in line:
                        attendance = line.split(':', 1)[-1].strip()
                    elif i + 1 < len(lines):
                        attendance = lines[i + 1].strip()

                elif 'stadium' in low or 'venue' in low:
                    if ':' in line:
                        venue = line.split(':', 1)[-1].strip()
                    elif i + 1 < len(lines):
                        nxt = lines[i + 1].strip()
                        if nxt and not any(
                            k in nxt.lower()
                            for k in ['attendance', 'duration', 'record', 'time of game']
                        ):
                            venue = nxt

        except Exception:
            pass

        try:
            driver.execute_script("arguments[0].click();", btn)
        except Exception:
            pass
        time.sleep(0.2)

    except NoSuchElementException:
        pass
    except Exception:
        pass

    return attendance, venue



In [6]:
### start time recovery (selenium, batch after season scrape)
# visits box score pages for games still showing TBA/TBD
# done as a separate pass so we don't navigate away from the schedule page mid-loop

NEEDS_START_TIME_RECOVERY = {'TBA', 'TBD', 'Postponed', '', '1 HR After GM2'}


def _get_start_time_selenium(driver, url):
    if not url or str(url).lower().endswith('.pdf'):
        return None
    try:
        driver.get(url)
        time.sleep(3.0)

        # try to click the "Game Information" expansion panel
        panel_clicked = False
        button_xpaths = [
            "//span[contains(@class, 's-expansion-panel__header-text') and "
            "contains(normalize-space(.), 'Game Information')]"
            "/ancestor::button",
            "//*[contains(normalize-space(.), 'Game Information') and "
            "(self::button or self::div[contains(@class, 'panel')] or "
            "self::span[contains(@class, 'panel')])]",
            "//*[contains(normalize-space(text()), 'Game Information')]"
            "/ancestor-or-self::button",
        ]
        for xpath in button_xpaths:
            try:
                panel_btn = driver.find_element(By.XPATH, xpath)
                driver.execute_script(
                    "arguments[0].scrollIntoView({block: 'center'});", panel_btn
                )
                time.sleep(0.3)
                driver.execute_script("arguments[0].click();", panel_btn)
                time.sleep(1.5)
                panel_clicked = True
                break
            except (NoSuchElementException, StaleElementReferenceException):
                continue
            except Exception:
                continue

        # strategy 1: CSS selectors for bold label + sibling
        if panel_clicked:
            try:
                panel_selectors = [
                    ".s-glossary.game-information",
                    "[class*='game-information']",
                    ".s-expansion-panel__body",
                ]
                for sel in panel_selectors:
                    try:
                        labels = driver.find_elements(
                            By.CSS_SELECTOR,
                            f"{sel} .s-text-paragraph-bold"
                        )
                        for label in labels:
                            if 'Start Time' in label.text:
                                val = label.find_element(
                                    By.XPATH, "following-sibling::span"
                                ).text.strip()
                                if val and re.search(r'\d', val):
                                    return val
                    except Exception:
                        continue
            except Exception:
                pass

        # strategy 2: text from expanded panel
        if panel_clicked:
            try:
                for sel in ['.s-glossary.game-information',
                            "[class*='game-information']",
                            '.s-expansion-panel__body']:
                    try:
                        panel = driver.find_element(By.CSS_SELECTOR, sel)
                        text = panel.text
                        m = re.search(
                            r'Start Time[:\s]+([\d]{1,2}[:.]\d{2}\s*(?:AM|PM|am|pm|a\.m\.|p\.m\.))',
                            text, re.IGNORECASE,
                        )
                        if m:
                            return m.group(1).strip()
                        m = re.search(r'Start Time[:\s]+(\d{1,2}[:.]\d{2})', text, re.IGNORECASE)
                        if m:
                            t = m.group(1).strip().replace('.', ':')
                            hr = int(t.split(':')[0])
                            suffix = 'AM' if 10 <= hr < 12 else 'PM'
                            return f'{t} {suffix}'
                    except NoSuchElementException:
                        continue
            except Exception:
                pass

        # strategy 3: use textContent (includes hidden/collapsed text)
        try:
            all_text = driver.execute_script('return document.body.textContent || document.body.innerText')
            if all_text:
                m = re.search(
                    r'Start\s*Time[:\s]+([\d]{1,2}[:.]\d{2}\s*(?:AM|PM|am|pm|a\.m\.|p\.m\.))',
                    all_text, re.IGNORECASE,
                )
                if m:
                    t = m.group(1).strip()
                    t = t.replace('.', '').upper()
                    t = re.sub(r'(\d)(AM|PM)', r'\1 \2', t)
                    return t
                m = re.search(
                    r'Start\s*Time[:\s]+(\d{1,2}[:.]\d{2})',
                    all_text, re.IGNORECASE,
                )
                if m:
                    t = m.group(1).strip().replace('.', ':')
                    hr = int(t.split(':')[0])
                    suffix = 'AM' if 10 <= hr < 12 else 'PM'
                    return f'{t} {suffix}'
        except Exception:
            pass

        # strategy 4: search full page source HTML for JSON patterns
        try:
            page_src = driver.page_source
            if page_src:
                for pattern in [
                    r'"start_time"\s*:\s*"([^"]+)"',
                    r'"startTime"\s*:\s*"([^"]+)"',
                    r'"gameTime"\s*:\s*"([^"]+)"',
                    r'"start"\s*:\s*"(\d{1,2}[:.][\\d]{2}\s*(?:AM|PM|am|pm)[^"]*)"',
                ]:
                    m = re.search(pattern, page_src, re.IGNORECASE)
                    if m:
                        t = m.group(1).strip()
                        if re.search(r'\d', t) and len(t) < 20:
                            t = t.replace('.', '').upper()
                            t = re.sub(r'(\d)(AM|PM)', r'\1 \2', t)
                            return t

                m = re.search(
                    r'Start\s*Time[:\s</span>]+([\d]{1,2}[:.]\d{2}\s*(?:AM|PM|am|pm))',
                    page_src, re.IGNORECASE,
                )
                if m:
                    t = m.group(1).strip().upper()
                    t = re.sub(r'(\d)(AM|PM)', r'\1 \2', t)
                    return t
        except Exception:
            pass

        # strategy 5: JS DOM traversal to find Game Information panel text
        try:
            panel_text = driver.execute_script("""
                var panels = document.querySelectorAll(
                    '.s-expansion-panel, [class*="expansion-panel"], [class*="game-info"]'
                );
                for (var i = 0; i < panels.length; i++) {
                    var tc = panels[i].textContent || '';
                    if (tc.indexOf('Start Time') !== -1 || tc.indexOf('start_time') !== -1) {
                        return tc;
                    }
                }
                var all = document.querySelectorAll('*');
                for (var j = 0; j < all.length; j++) {
                    var t = all[j].textContent || '';
                    if (t.indexOf('Start Time') !== -1 && t.length < 2000) {
                        return t;
                    }
                }
                return null;
            """)
            if panel_text:
                m = re.search(
                    r'Start\s*Time[:\s]+([\d]{1,2}[:.]\d{2}\s*(?:AM|PM|am|pm|a\.m\.|p\.m\.))',
                    panel_text, re.IGNORECASE,
                )
                if m:
                    t = m.group(1).strip()
                    t = t.replace('.', '').upper()
                    t = re.sub(r'(\d)(AM|PM)', r'\1 \2', t)
                    return t
                m = re.search(
                    r'Start\s*Time[:\s]+(\d{1,2}[:.]\d{2})',
                    panel_text, re.IGNORECASE,
                )
                if m:
                    t = m.group(1).strip().replace('.', ':')
                    hr = int(t.split(':')[0])
                    suffix = 'AM' if 10 <= hr < 12 else 'PM'
                    return f'{t} {suffix}'
        except Exception:
            pass

    except Exception:
        pass
    return None


def recover_start_times_batch(driver, games):
    recovered = 0
    for g in games:
        st = g.get('start_time') or ''
        if st in NEEDS_START_TIME_RECOVERY and g.get('box_score_link') and g.get('result'):
            print(f"      recovering start time for "
                  f"{g.get('date')} vs {g.get('opponent')} (current: '{st}')")
            try:
                bs = get_box_score_details(g['box_score_link'])
                if bs.get('start_time') and bs['start_time'] not in NEEDS_START_TIME_RECOVERY:
                    g['start_time'] = bs['start_time']
                    recovered += 1
                    print(f"      -> got {bs['start_time']} via requests")
                    continue
                else:
                    print(f"      -> requests returned {bs.get('start_time')!r}, trying selenium...")

                sel_time = _get_start_time_selenium(driver, g['box_score_link'])
                if sel_time and sel_time not in NEEDS_START_TIME_RECOVERY:
                    g['start_time'] = sel_time
                    recovered += 1
                    print(f"      -> got {sel_time} via selenium")
                else:
                    print(f"      -> FAILED (selenium returned: {sel_time!r})")
            except Exception as e:
                print(f"      warning: failed for {g.get('date')} vs {g.get('opponent')}: {str(e)[:200]}")
    print(f"  recovered {recovered} start time(s) in batch." if recovered else "  no start times recovered in batch.")
    return games



In [7]:
### game card data extraction
# reads all fields from a single schedule game card

def _safe_text(card, selector):
    try:
        return card.find_element(By.CSS_SELECTOR, selector).text.strip()
    except Exception:
        return None


def _safe_attr(card, selector, attr):
    try:
        return card.find_element(By.CSS_SELECTOR, selector).get_attribute(attr)
    except Exception:
        return None


def extract_game_data(driver, card, year, game_num, tournament=None, use_box_score=False):
    g = {
        'season': year,
        'game_number': game_num,
        'tournament': tournament,
        'unc_score': None,
        'opponent_score': None,
    }

    try:
        # basic fields from the schedule card
        opp_sel = "[data-test-id='s-game-card-standard__header-team-opponent-link']"
        g['opponent'] = _safe_text(card, opp_sel)

        date_text = _safe_text(card, "[data-test-id='s-game-card-standard__header-game-date-details']")
        g['date'] = date_text.replace('\n', ' ') if date_text else None

        try:
            time_el = card.find_element(
                By.CSS_SELECTOR,
                "[data-test-id='s-game-card-standard__header-game-time']",
            )
            spans = time_el.find_elements(By.CSS_SELECTOR, 'span.s-text-paragraph-small')
            g['start_time'] = spans[0].text.strip() if spans else None
        except Exception:
            g['start_time'] = None

        card_venue = _valid_venue(
            _safe_text(card, "[data-test-id='s-game-card-facility-and-location__game-facility-title-link']")
        )
        g['venue'] = card_venue if not use_box_score else None

        g['location_city'] = _safe_text(
            card,
            "[data-test-id='s-game-card-facility-and-location__standard-location-details']",
        )

        g['result'] = _safe_text(
            card,
            "[data-test-id='s-game-card-standard__header-game-team-score']",
        )

        promo = _safe_text(card, "[data-test-id='s-game-card-standard__header-promotion-btn']")
        g['promotion'] = promo if promo else None

        # box score link (needed for start time recovery and era 1 data)
        g['box_score_link'] = None
        try:
            for btn in card.find_elements(By.CSS_SELECTOR, '.s-game-card-game-link-button'):
                txt = (btn.text or '').lower()
                if any(kw in txt for kw in ('box', 'boxscore', 'stats')):
                    g['box_score_link'] = btn.get_attribute('href')
                    break
        except Exception:
            pass

        g['attendance'] = None

        # era 1 (1999-2016): pull data from the stats page
        if use_box_score:
            if g.get('box_score_link'):
                info = get_box_score_details(g['box_score_link'])

                g['attendance'] = _clean_attendance(info.get('attendance'))

                stats_venue = _valid_venue(info.get('venue'))
                if stats_venue:
                    g['venue'] = stats_venue
                elif card_venue:
                    g['venue'] = card_venue

                if not g.get('location_city') and info.get('location_city'):
                    g['location_city'] = info['location_city']

                if _needs_result_fix(g.get('result')) and info.get('result'):
                    g['result'] = info['result']
                if g.get('result') and g.get('unc_score') is None:
                    m_sc = re.search(r'([WLT]),?\s*(\d+)\s*-\s*(\d+)', str(g['result']))
                    if m_sc:
                        g['unc_score'] = m_sc.group(2)
                        g['opponent_score'] = m_sc.group(3)
                if info.get('unc_score') is not None and g.get('unc_score') is None:
                    g['unc_score'] = info['unc_score']
                    g['opponent_score'] = info.get('opponent_score')

                if not g.get('opponent') and info.get('opponent'):
                    g['opponent'] = info['opponent']

                if (not g.get('date') or str(g['date']).strip() == '') and info.get('date'):
                    g['date'] = info['date']

                if (not g.get('start_time') or
                        g['start_time'] in ('TBA', 'TBD', 'Postponed', '', '1 HR After GM2')):
                    if info.get('start_time'):
                        g['start_time'] = info['start_time']

        # era 2 (2017+): pull attendance/venue from game info dropdown, box score as fallback
        else:
            att, gi_venue = get_game_info_details(driver, card)
            g['attendance'] = _clean_attendance(att)

            gi_venue_clean = _valid_venue(gi_venue)
            if gi_venue_clean:
                g['venue'] = gi_venue_clean

            if g.get('box_score_link'):
                need_fallback = (
                    not g.get('venue')
                    or _needs_result_fix(g.get('result'))
                    or not g.get('attendance')
                    or not g.get('start_time')
                    or g.get('start_time') in ('TBA', 'TBD', 'Postponed', '', '1 HR After GM2')
                    or not g.get('date')
                    or str(g.get('date', '')).strip() == ''
                )
                if need_fallback:
                    bs = get_box_score_details(g['box_score_link'])

                    if not g.get('venue'):
                        g['venue'] = _valid_venue(bs.get('venue'))
                    if not g.get('attendance'):
                        g['attendance'] = _clean_attendance(bs.get('attendance'))
                    if (not g.get('start_time') or
                            g['start_time'] in ('TBA', 'TBD', 'Postponed', '', '1 HR After GM2')):
                        if bs.get('start_time'):
                            g['start_time'] = bs['start_time']
                    if (not g.get('date') or str(g.get('date', '')).strip() == ''):
                        if bs.get('date'):
                            g['date'] = bs['date']
                    if _needs_result_fix(g.get('result')) and bs.get('result'):
                        g['result'] = bs['result']
                    if g.get('result') and g.get('unc_score') is None:
                        m_sc = re.search(r'([WLT]),?\s*(\d+)\s*-\s*(\d+)', str(g['result']))
                        if m_sc:
                            g['unc_score'] = m_sc.group(2)
                            g['opponent_score'] = m_sc.group(3)
                    if bs.get('unc_score') is not None and g.get('unc_score') is None:
                        g['unc_score'] = bs['unc_score']
                        g['opponent_score'] = bs.get('opponent_score')
                    if not g.get('opponent') and bs.get('opponent'):
                        g['opponent'] = bs['opponent']

        # make sure scores are always set when we have a result string
        if g.get('result') and (g.get('unc_score') is None or g.get('opponent_score') is None):
            m = re.search(r'([WLT]),?\s*(\d+)\s*-\s*(\d+)', str(g['result']))
            if m:
                g['unc_score'] = m.group(2)
                g['opponent_score'] = m.group(3)

        # clean opponent - strip rankings and extract tournament names
        if g.get('opponent'):
            clean_opp, tourn_from_opp = _extract_tournament_from_opponent(g['opponent'])
            if tourn_from_opp and not g.get('tournament'):
                g['tournament'] = tourn_from_opp
            g['opponent'] = clean_opp
            g['opponent'] = _clean_opponent(g['opponent'])

        # clean location - extract tournament names from parentheticals
        if g.get('location_city'):
            clean_loc, tourn_from_loc = _extract_tournament_from_location(g['location_city'])
            if tourn_from_loc and not g.get('tournament'):
                g['tournament'] = tourn_from_loc
            g['location_city'] = clean_loc

        # venue fallback - infer from city if nothing else worked
        if not g.get('venue') and g.get('location_city'):
            fb = _venue_fallback(g['location_city'], year)
            if fb:
                g['venue'] = fb

        return g

    except Exception as e:
        print(f"    error extracting game {game_num}: {e}")
        return g



In [8]:
### season scraper
# loads one season's schedule page and scrapes every game card

def scrape_season(driver, year):
    url = f'https://goheels.com/sports/baseball/schedule/{year}'
    use_box_score = (1999 <= year <= 2016)
    mode = 'era 1 (box score)' if use_box_score else 'era 2 (game info dropdown)'
    print(f'\n{"="*60}')
    print(f'{year} - {mode}')
    print(f'{"="*60}')

    try:
        driver.get(url)
        time.sleep(3)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CLASS_NAME, 's-game-card'))
        )

        games = []
        tournament = None
        n = 0

        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(1)
        driver.execute_script('window.scrollTo(0, 0);')
        time.sleep(0.5)

        container = driver.find_element(
            By.CSS_SELECTOR,
            "[data-test-id='schedule-view-type-list__root']",
        )

        for el in container.find_elements(By.XPATH, './*'):
            try:
                try:
                    tournament = el.find_element(
                        By.CSS_SELECTOR,
                        "[data-test-id='schedule-view-type-list__tournament-title']",
                    ).text.strip()
                    continue
                except NoSuchElementException:
                    pass

                cls = el.get_attribute('class') or ''
                if 's-game-card' in cls:
                    n += 1
                    g = extract_game_data(driver, el, year, n, tournament, use_box_score)
                    games.append(g)

                    if n <= 5 or n % 10 == 0:
                        print(
                            f"  {n:3d}: {g.get('date','?'):20s} {g.get('opponent','?'):25s} | "
                            f"result={str(g.get('result','?')):12s} "
                            f"att={str(g.get('attendance','N/A')):6s} "
                            f"venue={str(g.get('venue','N/A'))[:40]}"
                        )

            except StaleElementReferenceException:
                continue
            except Exception:
                continue

        print(f'{year} done: {len(games)} games scraped')

        # batch recover start times for TBA/TBD games
        try:
            games = recover_start_times_batch(driver, games)
        except Exception as e:
            print(f'  warning: batch start time recovery failed: {str(e)[:200]}')

        # second-pass score recovery for era 1 games
        if use_box_score:
            for g in games:
                if g.get('result') is None and g.get('box_score_link') and g.get('attendance'):
                    try:
                        info = get_box_score_details(g['box_score_link'])
                        if info.get('result'):
                            g['result'] = info['result']
                            g['unc_score'] = info.get('unc_score')
                            g['opponent_score'] = info.get('opponent_score')
                            print(f"      -> score recovered (2nd pass): {info['result']} "
                                  f"({g.get('date')} vs {g.get('opponent')})")
                    except Exception as e:
                        print(f"      warning: score recovery failed for "
                              f"{g.get('date')} vs {g.get('opponent')}: {str(e)[:100]}")

        return games, driver

    except TimeoutException:
        print(f'  timeout loading {year} schedule page. no games found.')
        return [], driver

    except WebDriverException as e:
        msg = str(e).lower()
        if any(k in msg for k in ['invalid session', 'session not created', 'no such session']):
            print('  browser session crashed. restarting...')
            try:
                driver.quit()
            except Exception:
                pass
            driver = setup_driver()
            try:
                return scrape_season(driver, year)
            except Exception:
                return [], driver
        print(f'browser error for {year}: {str(e)[:200]}')
        return [], driver

    except Exception as e:
        print(f'error scraping {year}: {str(e)[:200]}')
        return [], driver



In [9]:
### main orchestrator - scrapes all seasons and saves raw CSV

def standardize(df):
    for c in STANDARD_COLUMNS:
        if c not in df.columns:
            df[c] = None
    return df[STANDARD_COLUMNS]


def scrape_all_seasons(start_year=1999, end_year=2026):
    start_year = max(1999, int(start_year))
    end_year = int(end_year)

    all_games = []
    driver = None

    try:
        driver = setup_driver()
        print(f'starting scrape: {start_year} to {end_year}\n')

        for year in range(start_year, end_year + 1):
            season_games, driver = scrape_season(driver, year)
            all_games.extend(season_games)
            time.sleep(1.5)

        df = standardize(pd.DataFrame(all_games))
        filename = 'unc_baseball_raw.csv'
        df.to_csv(filename, index=False)

        print(f'\n{"="*60}')
        print(f'SCRAPE COMPLETE')
        print(f'{"="*60}')
        print(f'saved raw output to: {filename}')
        print(f'total games scraped: {len(df)}')
        print(f'attendance coverage: {df["attendance"].notna().sum()}/{len(df)} '
              f'({df["attendance"].notna().mean():.0%})')
        print(f'venue coverage: {df["venue"].notna().sum()}/{len(df)} '
              f'({df["venue"].notna().mean():.0%})')
        print(f'\ngames per season:')
        print(df['season'].value_counts().sort_index().to_string())
        print(f'{"="*60}')
        return df

    finally:
        if driver:
            try:
                driver.quit()
            except Exception:
                pass
            print('browser closed.')


In [10]:
### post-processing cleanup
# standardizes names, fixes formatting, fills gaps, drops helper columns

def post_process(df):
    df = df.copy()

    # 1. standardize venue names
    venue_map = {
        'Boshamer': 'Boshamer Stadium',
        'Boshamer stadium': 'Boshamer Stadium',
        'Boshmer Stadium': 'Boshamer Stadium',
        'Boshamer Stadiun': 'Boshamer Stadium',
        'Boshmaer Stadium': 'Boshamer Stadium',
        'Bosahmer Stadium': 'Boshamer Stadium',
        'Boashamer Stadium': 'Boshamer Stadium',
        'Durham Bulls Ath Pk': 'Durham Bulls Athletic Park',
        'Durham Bulls Ath Pk.': 'Durham Bulls Athletic Park',
        'Durham Bulls Ath Prk': 'Durham Bulls Athletic Park',
        'Durham Bulls Ath. Pk': 'Durham Bulls Athletic Park',
        'Bulls Athletic Park': 'Durham Bulls Athletic Park',
        'Durham, N.C.': 'Durham Bulls Athletic Park',
        'Clark-LeClair': 'Clark-LeClair Stadium',
        'Clark-LeClair Stad.': 'Clark-LeClair Stadium',
        'Dick Howser': 'Dick Howser Stadium',
        'Dick Howser Field': 'Dick Howser Stadium',
        'Doug Kingsmore Stad.': 'Doug Kingsmore Stadium',
        'Doug Kingsmore': 'Doug Kingsmore Stadium',
        'Kingsmore Stadium': 'Doug Kingsmore Stadium',
        'Russ Chandler': 'Mac Nease Park',
        'Russ Chandler Stdm.': 'Mac Nease Park',
        'R. Chandler Stadium': 'Mac Nease Park',
        'Chandler Stadium': 'Mac Nease Park',
        'Chander Stadium': 'Mac Nease Park',
        'Cost Field': 'Charles L. Cost Field',
        'Charles L Cost Field': 'Charles L. Cost Field',
        'Charles Cost Field': 'Charles L. Cost Field',
        'Couch Ballpark': 'David F. Couch Ballpark',
        'WF Baseball Park': 'David F. Couch Ballpark',
        'Salem Memorial': 'Salem Memorial Stadium',
        'Salem Memorial Stad.': 'Salem Memorial Stadium',
        'Salem Memorial Park': 'Salem Memorial Stadium',
        'Salem Memorial Stdm.': 'Salem Memorial Stadium',
        'UVa Baseball Stadium': 'Disharoon Park',
        'UVA Baseball Stadium': 'Disharoon Park',
        'UVa Baseball Field': 'Disharoon Park',
        'UVA Baseball Field': 'Disharoon Park',
        'UVa Stadium': 'Disharoon Park',
        'Gene Hooks Stadium': 'Gene Hooks Field',
        'Jackie Robinson': 'Jackie Robinson Stadium',
        'J. Robinson Stadium': 'Jackie Robinson Stadium',
        'Jackie Robinson Std.': 'Jackie Robinson Stadium',
        'Liberty Baseball': 'Liberty Baseball Stadium',
        'Springs Brooks': 'Springs Brooks Stadium',
        'Springs Brooks Stad.': 'Springs Brooks Stadium',
        'Bieden Field': 'Pete Bieden Field',
        'Pete Bieden Field': 'Pete Bieden Field',
        'UNCG Stadium': 'UNCG Baseball Stadium',
        'UNCG Bball Stadium': 'UNCG Baseball Stadium',
        'UNCG Baseball Stdm.': 'UNCG Baseball Stadium',
        'Frank Eck': 'Frank Eck Stadium',
        'Bob "Turtle" Smith': 'Bob Smith Stadium',
        'Sewell-Thomas Stad.': 'Sewell-Thomas Stadium',
        'Harrington Village': 'Harrington Field',
        'Pellagrini Diamond': 'Harrington Field',
        'Pellagrini Field': 'Harrington Field',
        'Alex Rodriguez Park': 'Mark Light Field',
        'Patterson Stadium': 'Jim Patterson Stadium',
        'EF at AUBP': 'English Field',
        'EF at Union Park': 'English Field',
        'R&M Hayes Stadium': 'Hayes Stadium',
        'Wide World of Sports': 'Disney Wide World of Sports',
        'Coastal Fed. Field': 'Coastal Federal Field',
        'Dail Park': 'Doak Field',
        'BUD METHENY': 'Bud Metheny Baseball Complex',
        'Davenport Field': 'Disharoon Park',
        'Davenport Field at Disharoon Park': 'Disharoon Park',
        'Louisville Slugger': 'Jim Patterson Stadium',
        'Louisville Slugger Fld.': 'Jim Patterson Stadium',
        'Slugger Field': 'Jim Patterson Stadium',
        'Truist Field': 'Truist Field',
        'Segra Stadium': 'Segra Stadium',
    }
    df['venue'] = df['venue'].map(lambda v: venue_map.get(v, v) if pd.notna(v) else v)

    def _split_bbt(row):
        if row['venue'] == 'BB&T Ballpark':
            if 'Charlottesville' in str(row.get('location_city') or ''):
                return 'Disharoon Park'
            return 'Truist Field'
        return row['venue']
    df['venue'] = df.apply(_split_bbt, axis=1)

    truist_cvl = (
        (df['venue'] == 'Truist Field')
        & df['location_city'].str.contains('Charlottesville', na=False)
    )
    df.loc[truist_cvl, 'venue'] = 'Disharoon Park'

    # 2. standardize opponent names
    opponent_map = {
        'No Carolina St': 'NC State',
        'N.C. State': 'NC State',
        'Nc Wilmington': 'UNC Wilmington',
        'UNC - Wilmington': 'UNC Wilmington',
        'UNC- Wilmington': 'UNC Wilmington',
        'UNCW': 'UNC Wilmington',
        'S. Carolina': 'South Carolina',
        'Westrn Carolina': 'Western Carolina',
        'W. Carolina': 'Western Carolina',
        'Appalachian St': 'Appalachian State',
        'App State': 'Appalachian State',
        'Va Commonwealth': 'Virginia Commonwealth',
        'VCU': 'Virginia Commonwealth',
        'Penn St': 'Penn State',
        'Michigan St': 'Michigan State',
        'Miami Fla': 'Miami',
        'Miami (FL)': 'Miami',
        'Miami (Fla.)': 'Miami',
        'Nc Greensboro': 'UNC Greensboro',
        'UNC-Greensboro': 'UNC Greensboro',
        'UNCG': 'UNC Greensboro',
        'UNC-G': 'UNC Greensboro',
        'UNC Asheville': 'UNC Asheville',
        'UNC-Asheville': 'UNC Asheville',
        'NC A&T': 'North Carolina A&T',
        'William &amp; Mary': 'William & Mary',
        'William &amp;amp; Mary': 'William & Mary',
        'Elon University': 'Elon',
        'ECU': 'East Carolina',
        'LeMoyne': 'Le Moyne',
        'Mt. Olive': 'Mount Olive',
        'St Francis': 'Saint Francis',
        'Saint Johns': "St. John's",
        'St Johns': "St. John's",
        'St. Johns': "St. John's",
        'Pitt': 'Pittsburgh',
        'Cal': 'California',
        'USC': 'Southern California',
        'UConn': 'Connecticut',
        'High Point University': 'High Point',
        'Liberty University': 'Liberty',
        'Texas Tech University': 'Texas Tech',
        'Dallas Baptist University': 'Dallas Baptist',
        'Presbyterian College': 'Presbyterian',
        'FGCU': 'Florida Gulf Coast',
    }
    df['opponent'] = df['opponent'].map(lambda v: opponent_map.get(v, v) if pd.notna(v) else v)

    for idx, row in df.iterrows():
        opp = row.get('opponent')
        if pd.notna(opp):
            opp = str(opp).strip()
            clean_opp, tourn = _extract_tournament_from_opponent(opp)
            if tourn and pd.isna(row.get('tournament')):
                df.loc[idx, 'tournament'] = tourn
            df.loc[idx, 'opponent'] = _clean_opponent(clean_opp)

    # 3. standardize tournament names
    tournament_map = {
        'ACC Tourney': 'ACC Tournament',
        'ACC Championships': 'ACC Championship',
        'Broadway at Beach': 'Broadway at Beach Tournament',
        'Broadway at Beach Tourn.': 'Broadway at Beach Tournament',
        'CambriaCollegeClassic': 'Cambria College Classic',
        'NCAA College World Series': 'College World Series',
        '2018 College World Series': 'College World Series',
        '2018 NCAA Chapel Hill Regional': 'NCAA Chapel Hill Regional',
        '2018 NCAA Chapel Hill Super Regional': 'NCAA Chapel Hill Super Regional',
        'NCAA Regional Championships': 'NCAA Regional',
    }
    df['tournament'] = df['tournament'].map(
        lambda v: tournament_map.get(v, v) if pd.notna(v) else v
    )

    # 4. clean location_city (extract embedded tournament names)
    for idx, row in df.iterrows():
        loc = row.get('location_city')
        if pd.notna(loc):
            clean_loc, tourn = _extract_tournament_from_location(str(loc))
            if tourn and pd.isna(row.get('tournament')):
                df.loc[idx, 'tournament'] = tourn
            df.loc[idx, 'location_city'] = clean_loc

    # 5. parse scores from result string
    def _parse_scores(row):
        if pd.notna(row.get('unc_score')):
            return row['unc_score'], row['opponent_score']
        result = str(row.get('result', '') or '')
        m = re.search(r'([WLT]),?\s*(\d+)\s*-\s*(\d+)', result)
        if m:
            return m.group(2), m.group(3)
        return row.get('unc_score'), row.get('opponent_score')

    scores = df.apply(_parse_scores, axis=1, result_type='expand')
    df['unc_score'] = scores[0]
    df['opponent_score'] = scores[1]

    # 6. fill missing location_city for Boshamer games
    for idx, row in df[df['location_city'].isna()].iterrows():
        if pd.notna(row.get('venue')) and 'Boshamer' in str(row['venue']):
            df.loc[idx, 'location_city'] = 'Chapel Hill, N.C.'

    # 7. standardize location_city format
    if 'location_city' in df.columns:
        df['location_city'] = df['location_city'].replace({
            'Atlanta': 'Atlanta, Ga.',
            'Los Angeles': 'Los Angeles, Calif.',
            'Maryland': 'College Park, Md.',
        })
        df['location_city'] = df['location_city'].replace({
            'Notre Dame, Ind.': 'South Bend, Ind.',
        })
        df['location_city'] = df['location_city'].replace({
            'Charlotte. N.C.': 'Charlotte, N.C.',
            'Durham, N.C': 'Durham, N.C.',
            'Chaple Hill, NC': 'Chapel Hill, N.C.',
            'Chapel Hill N.C.': 'Chapel Hill, N.C.',
            'Chapel Hill': 'Chapel Hill, N.C.',
            'Chapel Hill, NC': 'Chapel Hill, N.C.',
        })
        city_map = {
            'Durham, NC': 'Durham, N.C.',
            'Raleigh, NC': 'Raleigh, N.C.',
            'Charlotte, NC': 'Charlotte, N.C.',
            'Greenville, NC': 'Greenville, N.C.',
            'Wilmington, NC': 'Wilmington, N.C.',
            'Winston-Salem, NC': 'Winston-Salem, N.C.',
            'Buies Creek, NC': 'Buies Creek, N.C.',
            'Boone, NC': 'Boone, N.C.',
            'Clemson, SC': 'Clemson, S.C.',
            'Charleston, SC': 'Charleston, S.C.',
            'Columbia, S.C': 'Columbia, S.C.',
            'Charlottesville, VA': 'Charlottesville, Va.',
            'Blacksburg, VA': 'Blacksburg, Va.',
            'Salem, VA': 'Salem, Va.',
            'Lubbock, TX': 'Lubbock, Texas',
            'Lynchburg, VA': 'Lynchburg, Va.',
            'Tallahassee, FL': 'Tallahassee, Fla.',
            'Coral Gables, FL': 'Coral Gables, Fla.',
            'College Park, MD': 'College Park, Md.',
            'Brighton, MA': 'Brighton, Mass.',
            'Chestnut Hill, MA': 'Chestnut Hill, Mass.',
            'Louisville, KY': 'Louisville, Ky.',
            'Pittsburgh, PA': 'Pittsburgh, Pa.',
            'Fresno, CA': 'Fresno, Calif.',
            'Palo Alto, CA': 'Palo Alto, Calif.',
        }
        df['location_city'] = df['location_city'].map(
            lambda v: city_map.get(v, v) if pd.notna(v) else v
        )

    # 8. normalize start times
    def _normalize_start_time(t):
        if pd.isna(t):
            return t
        t = str(t).strip()
        if t in ('Canceled', 'Postponed', 'TBA', 'TBD', '', '1 HR After GM2'):
            return t
        t = re.sub(r'\s*(ET|CT|PT|MT)\s*$', '', t, flags=re.IGNORECASE).strip()
        t = re.sub(r'^Approx\.?\s*', '', t, flags=re.IGNORECASE).strip()
        if t.lower() in ('noon', '12 noon'):
            return '12:00 PM'
        t = t.replace('a.m.', 'AM').replace('p.m.', 'PM')
        t = t.replace('A.M.', 'AM').replace('P.M.', 'PM')
        t = re.sub(r'\s+', ' ', t).strip()
        m = re.match(r'^(\d{1,2})\s+(AM|PM|am|pm)$', t)
        if m:
            t = f"{m.group(1)}:00 {m.group(2).upper()}"
        t = re.sub(r'\b(am|pm)\b', lambda x: x.group().upper(), t, flags=re.IGNORECASE)
        t = re.sub(r'(\d+):\s+(\d+)', r'\1:\2', t)
        t = re.sub(r'(\d)(AM|PM)', r'\1 \2', t)
        m2 = re.match(r'^(\d{1,2}):(\d{2})\s+AM$', t)
        if m2 and int(m2.group(1)) < 7:
            t = f"{m2.group(1)}:{m2.group(2)} PM"
        m3 = re.match(r'^(11):(\d{2})\s+PM$', t)
        if m3:
            t = f"{m3.group(1)}:{m3.group(2)} AM"
        return t

    df['start_time'] = df['start_time'].map(_normalize_start_time)

    # 9. normalize result format (ensure "W, N-N" with comma)
    def _normalize_result(r):
        if pd.isna(r):
            return r
        r = str(r).strip()
        m = re.match(r'^([WLT])\s+(\d+-\d+)$', r)
        if m:
            return f"{m.group(1)}, {m.group(2)}"
        return r

    df['result'] = df['result'].map(_normalize_result)

    # 10. fix score mismatches (result string is authoritative)
    for idx, row in df.iterrows():
        if pd.notna(row.get('result')):
            m_sc = re.match(r'^([WLT]), (\d+)-(\d+)$', str(row['result']))
            if m_sc:
                correct_unc = int(m_sc.group(2))
                correct_opp = int(m_sc.group(3))
                csv_unc = int(row['unc_score']) if pd.notna(row.get('unc_score')) else None
                csv_opp = int(row['opponent_score']) if pd.notna(row.get('opponent_score')) else None
                if csv_unc != correct_unc or csv_opp != correct_opp:
                    df.loc[idx, 'unc_score'] = correct_unc
                    df.loc[idx, 'opponent_score'] = correct_opp

    # 10b. hardcoded score corrections for known scrape errors
    score_field_corrections = {
        (2009, 10): (4, 5),
        (2009, 51): (4, 9),
    }
    for (szn, gnum), (correct_unc, correct_opp) in score_field_corrections.items():
        mask = (df['season'] == szn) & (df['game_number'] == gnum)
        df.loc[mask, 'unc_score'] = correct_unc
        df.loc[mask, 'opponent_score'] = correct_opp

    # 11. hardcoded start times for games where the website never had the time
    hardcoded_times = {
        (1999, 31): '7:00 PM',
        (1999, 32): '7:00 PM',
        (1999, 60): '6:00 PM',
        (1999, 61): '6:00 PM',
        (1999, 62): '6:00 PM',
        (2022, 47): '2:25 PM',
        (2023, 30): '3:20 PM',
    }
    for (szn, gnum), t in hardcoded_times.items():
        mask = (df['season'] == szn) & (df['game_number'] == gnum)
        df.loc[mask, 'start_time'] = t

    # 11b. doubleheader attendance sharing
    for (season, date), group in df.groupby(['season', 'date']):
        if len(group) >= 2:
            available = group['attendance'].dropna()
            if 0 < len(available) < len(group):
                fill_val = available.iloc[0]
                df.loc[group.index, 'attendance'] = (
                    group['attendance'].fillna(fill_val)
                )

    # 12. remove known duplicate (2007 game 33)
    df = df[~((df['season'] == 2007) & (df['game_number'] == 33))]
    mask_2007_after = (df['season'] == 2007) & (df['game_number'] >= 33)
    df.loc[mask_2007_after, 'game_number'] = df.loc[mask_2007_after, 'game_number'] - 1

    # 13. drop helper columns and fields we don't need
    drop_cols = ['box_score_link', 'location_city']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # 14. filter out incomplete rows
    # for 2026, keep all scheduled games even if they haven't been played yet
    before = len(df)
    is_2026 = (df['season'] == 2026)
    df = df[is_2026 | df['date'].notna()]
    df = df[is_2026 | df['opponent'].notna()]
    df = df[is_2026 | df['result'].notna()]
    df = df[is_2026 | ~df['start_time'].isin(['TBA', 'Postponed', ''])]
    df = df[is_2026 | df['start_time'].notna()]
    df = df.reset_index(drop=True)
    print(f"  filtered out {before - len(df)} incomplete rows, {len(df)} games remaining.")

    return df



In [11]:
### test single season (uncomment to run)

# df_test = scrape_all_seasons(2025, 2025)
# df_test


In [12]:
### full scrape

df = scrape_all_seasons(1999, 2026)


starting scrape: 1999 to 2026


1999 - era 1 (box score)
    1: Feb 12(Fri)          East Carolina             | result=W, 8-3       att=697    venue=Riley Park
    2: Feb 13(Sat)          Va Commonwealth           | result=W, 8-0       att=583    venue=Riley Park
    3: Feb 14(Sun)          The Citadel               | result=W, 8-5       att=664    venue=Riley Park
    4: Feb 19(Fri)          Seton Hall                | result=W, 11-3      att=None   venue=Boshamer Stadium
    5: Feb 20(Sat)          Seton Hall                | result=W, 9-3       att=469    venue=Boshamer Stadium
   10: Feb 28(Sun)          Temple                    | result=W, 8-1       att=925    venue=Boshamer Stadium
   20: Mar 17(Wed)          Appalachian St            | result=W, 4-3       att=1069   venue=Boshamer Stadium
   30: Mar 31(Wed)          Davidson                  | result=W, 11-6      att=407    venue=Boshamer Stadium
   40: Apr 16(Fri)          Virginia                  | result=W, 10-9      att=8

In [13]:
### post-process and save final CSV

df_clean = post_process(df)
df_clean.to_csv('unc_baseball_final.csv', index=False)
print(f'saved {len(df_clean)} games to unc_baseball_final.csv')


  filtered out 90 incomplete rows, 1717 games remaining.
saved 1717 games to unc_baseball_final.csv


C:\Users\Adam\AppData\Local\Temp\ipykernel_24296\1345417838.py:355: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[is_2026 | df['opponent'].notna()]
C:\Users\Adam\AppData\Local\Temp\ipykernel_24296\1345417838.py:356: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[is_2026 | df['result'].notna()]
C:\Users\Adam\AppData\Local\Temp\ipykernel_24296\1345417838.py:357: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[is_2026 | ~df['start_time'].isin(['TBA', 'Postponed', ''])]
C:\Users\Adam\AppData\Local\Temp\ipykernel_24296\1345417838.py:358: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[is_2026 | df['start_time'].notna()]
